In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Github Data Transformation - Bronze to Silver

This notebook processes device mapping and server performance data from the Bronze layer and loads it into the Silver layer with data quality checks and transformations.

## Load Bronze Data

In [0]:
mapping_df = spark.read.format('delta') \
    .load('/Volumes/telecom_catalog/default/bronze/Github_data/device_mapping/') \

server_df = spark.read.format('delta') \
    .load('/Volumes/telecom_catalog/default/bronze/Github_data/Server_performance_logs/')

# Processing Device Mapping Data

In [0]:
mapping_df.display()

### 1. Data Profiling and Data Quality Checks

In [0]:
print("Total Rows :", mapping_df.count())

mapping_df.groupBy("device_id").count().filter("count > 1").show()

mapping_df.groupBy("host_id").count().filter("count > 1").show()

mapping_df.groupBy("api_device_id").count().filter("count > 1").show()

Total Rows : 901
+---------+-----+
|device_id|count|
+---------+-----+
+---------+-----+

+-------+-----+
|host_id|count|
+-------+-----+
+-------+-----+

+-------------+-----+
|api_device_id|count|
+-------------+-----+
+-------------+-----+



## 2. Handling Nulls and Removing Duplicates

In [0]:

mapping_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in mapping_df.columns
]).show()

+-------------+---------+-------+
|api_device_id|device_id|host_id|
+-------------+---------+-------+
|            0|        0|      0|
+-------------+---------+-------+



In [0]:
mapping_df = mapping_df.dropDuplicates()

## 3. Standardizing Data


In [0]:
mapping_df = (
    mapping_df
    .withColumn("device_id", upper(trim(col("device_id"))))
    .withColumn("host_id", upper(trim(col("host_id"))))
    .withColumn("api_device_id", upper(trim(col("api_device_id"))))
)

## 4. Save to Silver Layer

In [0]:
(
    mapping_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save('/Volumes/telecom_catalog/default/silver/Github_data/device_mapping/')
)


# Processing Server Performance Data

## 1. Data Profiling and Duplicates handling

In [0]:
server_df.printSchema()


root
 |-- host_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- disk_usage: double (nullable = true)
 |-- network_errors: double (nullable = true)
 |-- uptime_hours: integer (nullable = true)
 |-- log_time: string (nullable = true)



In [0]:
print("Total Rows :", server_df.count())

Total Rows : 1225


In [0]:
server_df.groupBy("host_id", "log_time") \
         .count() \
         .filter("count > 1") \
         .show()

+-------+-------------------+-----+
|host_id|           log_time|count|
+-------+-------------------+-----+
|SRV_113|2026-05-26 10:13:00|    2|
|SRV_121|2026-05-26 10:21:00|    2|
|SRV_100|2026-05-26 10:00:00|    2|
|SRV_108|2026-05-26 10:08:00|    2|
|SRV_110|2026-05-26 10:10:00|    2|
|SRV_116|2026-05-26 10:16:00|    2|
|SRV_120|2026-05-26 10:20:00|    2|
|SRV_104|2026-05-26 10:04:00|    2|
|SRV_122|2026-05-26 10:22:00|    2|
|SRV_114|2026-05-26 10:14:00|    2|
|SRV_105|2026-05-26 10:05:00|    2|
|SRV_107|2026-05-26 10:07:00|    2|
|SRV_124|2026-05-26 10:24:00|    2|
|SRV_102|2026-05-24 10:00:00|    2|
|SRV_123|2026-05-26 10:23:00|    2|
|SRV_112|2026-05-26 10:12:00|    2|
|SRV_111|2026-05-26 10:11:00|    2|
|SRV_118|2026-05-26 10:18:00|    2|
|SRV_103|       INVALID_DATE|    2|
|SRV_106|2026-05-26 10:06:00|    2|
+-------+-------------------+-----+
only showing top 20 rows


In [0]:
server_df = server_df.dropDuplicates(["host_id", "log_time"])

In [0]:
server_df.groupBy("host_id", "log_time") \
         .count() \
         .filter("count > 1") \
         .show()

+-------+--------+-----+
|host_id|log_time|count|
+-------+--------+-----+
+-------+--------+-----+



## 2. Standardizing data

In [0]:
server_df = (
    server_df
    .withColumn("host_id", upper(trim(col("host_id"))))
    .withColumn("region", initcap(trim(col("region"))))
)

## 3. Convert log_time to Timestamp

In [0]:
server_df = server_df.withColumn(
    "log_time",
    try_to_timestamp("log_time", lit("yyyy-MM-dd HH:mm:ss"))
)

In [0]:
server_df.select(
    count(when(col("log_time").isNull(), True)).alias("invalid_dates")
).show()

+-------------+
|invalid_dates|
+-------------+
|           45|
+-------------+



In [0]:
server_df.filter(col("log_time").isNull()).display(50, truncate=False)

host_id,region,disk_usage,network_errors,uptime_hours,log_time
SRV_662,Delhi,82.58,5.0,767,null
SRV_701,Hyderabad,68.94,0.0,1132,null
SRV_765,Chennai,92.54,9.0,799,null
SRV_1077,Chennai,54.2,null,698,null
SRV_1227,Hyderabad,62.25,5.0,1828,null
SRV_254,Chennai,37.44,null,638,null
SRV_430,Chennai,87.48,8.0,1733,null
SRV_621,Delhi,150.0,1.0,157,null
SRV_687,Pune,43.1,0.0,1403,null
SRV_1122,Mumbai,84.64,null,349,null


## 4. Handling Nulls

In [0]:
server_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in server_df.columns
]).show()

+-------+------+----------+--------------+------------+--------+
|host_id|region|disk_usage|network_errors|uptime_hours|log_time|
+-------+------+----------+--------------+------------+--------+
|      0|     0|        42|           614|           0|      45|
+-------+------+----------+--------------+------------+--------+



#### disk_usage → Keep NULL

A missing disk usage means the server didn't report it. Replacing it with 0 would incorrectly indicate the disk is empty.

#### network_errors → Replace with 0

This depends on your business assumption. If a missing value means "no network errors reported," then replacing with 0 is reasonable. Since this is my own generated dataset, this assumption is acceptable.

In [0]:
server_df = server_df.withColumn(
    "network_errors",
    when(col("network_errors").isNull(), 0)
    .otherwise(col("network_errors"))
)

#### log_time → Keep NULL

validation_reason = Invalid Log Time

is_valid = False

In [0]:
server_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in server_df.columns
]).show()

+-------+------+----------+--------------+------------+--------+
|host_id|region|disk_usage|network_errors|uptime_hours|log_time|
+-------+------+----------+--------------+------------+--------+
|      0|     0|        42|             0|           0|      45|
+-------+------+----------+--------------+------------+--------+



## 5. Data Validation

In [0]:

server_df = (
    server_df
    .withColumn(
        "validation_reason",
        when(col("log_time").isNull(), "Invalid Log Time")
        .when(col("disk_usage").isNull(), "Disk Usage Missing")
        .when((col("disk_usage") < 0) | (col("disk_usage") > 100), "Invalid Disk Usage")
        .when(col("network_errors") < 0, "Invalid Network Errors")
        .when(col("uptime_hours") < 0, "Invalid Uptime")
        .otherwise("Valid")
    )
)

In [0]:
server_df = server_df.withColumn(
    "is_valid",
    when(col("validation_reason") == "Valid", True)
    .otherwise(False)
)

In [0]:
server_df.groupBy("validation_reason").count().show()

server_df.groupBy("is_valid").count().show()

+------------------+-----+
| validation_reason|count|
+------------------+-----+
|Invalid Disk Usage|  134|
|             Valid|  982|
|  Invalid Log Time|   45|
|Disk Usage Missing|   39|
+------------------+-----+

+--------+-----+
|is_valid|count|
+--------+-----+
|   false|  218|
|    true|  982|
+--------+-----+



## 6. Add Audit Columns

In [0]:

server_df = (
    server_df
    .withColumn("silver_load_time", current_timestamp())
    .withColumn("source_system", lit("Server Performance CSV"))
)

## 7. Save to Silver Layer

In [0]:
(
    server_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("/Volumes/telecom_catalog/default/silver/Github_data/Server_performance")
)